# Chapter 3: Multiagent Search

```{admonition} Learning Objectives
:class: tip
- Understand AND-OR search trees
- Implement Minimax algorithm
- Optimize with Alpha-Beta pruning
- Apply Monte Carlo Tree Search
- Build game-playing agents
```

## 3.1 Introduction

Multiagent environments require handling **contingencies** - situations where the agent cannot fully control outcomes.

### Key Differences

| Single-Agent | Multi-Agent |
|--------------|-------------|
| OR nodes only | AND and OR nodes |
| Single path | Multiple paths |
| Full control | Uncertainty |

In [ ]:
import numpy as np
import math
import random
from typing import List, Optional
import time

print('Libraries imported')

## 3.2 Game State Representation

In [ ]:
class GameState:
    def get_legal_actions(self, player):
        raise NotImplementedError
    
    def get_successor(self, action):
        raise NotImplementedError
    
    def is_terminal(self) -> bool:
        raise NotImplementedError
    
    def utility(self, player) -> float:
        raise NotImplementedError
    
    def current_player(self) -> int:
        raise NotImplementedError

print('GameState base class defined')

## 3.3 Minimax Algorithm

Minimax assumes optimal play from both players.

$$f(n) = \begin{cases}
\max_{s} f(s) & \text{if MAX} \\
\min_{s} f(s) & \text{if MIN}
\end{cases}$$

In [ ]:
def minimax(state, depth, maximizing_player, eval_fn=None, max_depth=float('inf')):
    if state.is_terminal():
        return state.utility(state.current_player()), None
    
    if depth >= max_depth:
        if eval_fn:
            return eval_fn(state), None
        return 0, None
    
    if maximizing_player:
        max_value = -float('inf')
        best_action = None
        for action in state.get_legal_actions(state.current_player()):
            successor = state.get_successor(action)
            value, _ = minimax(successor, depth + 1, False, eval_fn, max_depth)
            if value > max_value:
                max_value = value
                best_action = action
        return max_value, best_action
    else:
        min_value = float('inf')
        best_action = None
        for action in state.get_legal_actions(state.current_player()):
            successor = state.get_successor(action)
            value, _ = minimax(successor, depth + 1, True, eval_fn, max_depth)
            if value < min_value:
                min_value = value
                best_action = action
        return min_value, best_action

print('Minimax implemented')

## 3.4 Alpha-Beta Pruning

Eliminates irrelevant branches without affecting minimax value.

- Alpha: Best value for MAX
- Beta: Best value for MIN
- Prune when alpha >= beta

In [ ]:
def alpha_beta(state, depth, alpha, beta, maximizing_player, eval_fn=None, max_depth=float('inf')):
    if state.is_terminal():
        return state.utility(state.current_player()), None
    
    if depth >= max_depth:
        if eval_fn:
            return eval_fn(state), None
        return 0, None
    
    if maximizing_player:
        value = -float('inf')
        best_action = None
        for action in state.get_legal_actions(state.current_player()):
            successor = state.get_successor(action)
            child_value, _ = alpha_beta(successor, depth + 1, alpha, beta, False, eval_fn, max_depth)
            if child_value > value:
                value = child_value
                best_action = action
            alpha = max(alpha, value)
            if beta <= alpha:
                break
        return value, best_action
    else:
        value = float('inf')
        best_action = None
        for action in state.get_legal_actions(state.current_player()):
            successor = state.get_successor(action)
            child_value, _ = alpha_beta(successor, depth + 1, alpha, beta, True, eval_fn, max_depth)
            if child_value < value:
                value = child_value
                best_action = action
            beta = min(beta, value)
            if beta <= alpha:
                break
        return value, best_action

print('Alpha-Beta pruning implemented')

## 3.5 Tic-Tac-Toe Example

In [ ]:
class TicTacToe(GameState):
    def __init__(self, board=None, player=1):
        if board is None:
            self.board = np.zeros((3, 3), dtype=int)
        else:
            self.board = board.copy()
        self.player = player
    
    def get_legal_actions(self, player):
        return [(i, j) for i in range(3) for j in range(3) if self.board[i, j] == 0]
    
    def get_successor(self, action):
        new_state = TicTacToe(self.board, -self.player)
        new_state.board[action] = self.player
        return new_state
    
    def is_terminal(self) -> bool:
        return self.get_winner() is not None or len(self.get_legal_actions(self.player)) == 0
    
    def get_winner(self):
        for i in range(3):
            if abs(self.board[i, :].sum()) == 3:
                return self.board[i, 0]
        for j in range(3):
            if abs(self.board[:, j].sum()) == 3:
                return self.board[0, j]
        if abs(np.trace(self.board)) == 3:
            return self.board[0, 0]
        if abs(np.trace(np.fliplr(self.board))) == 3:
            return self.board[0, 2]
        return None
    
    def utility(self, player) -> float:
        winner = self.get_winner()
        if winner == player:
            return 1.0
        elif winner == -player:
            return -1.0
        return 0.0
    
    def current_player(self) -> int:
        return self.player

print('Tic-Tac-Toe implemented')

In [ ]:
game = TicTacToe()
print('Testing Alpha-Beta on Tic-Tac-Toe...')
value, action = alpha_beta(game, 0, -float('inf'), float('inf'), True)
print(f'Best move: {action}, Value: {value}')

## 3.6 Monte Carlo Tree Search

MCTS learns from experience via simulations.

### UCT Formula

$$UCT(n) = \frac{w_n}{n_n} + c\sqrt{\frac{\ln N_n}{n_n}}$$

In [ ]:
class MCTSNode:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = []
        self.visits = 0
        self.wins = 0
    
    def is_fully_expanded(self):
        return len(self.children) == len(self.state.get_legal_actions(self.state.current_player()))
    
    def best_child(self, c=1.41):
        return max(self.children, key=lambda node: 
                  node.wins / node.visits + c * math.sqrt(math.log(self.visits) / node.visits))
    
    def expand(self):
        tried_actions = [child.action for child in self.children]
        legal_actions = self.state.get_legal_actions(self.state.current_player())
        untried = [a for a in legal_actions if a not in tried_actions]
        action = random.choice(untried)
        next_state = self.state.get_successor(action)
        child = MCTSNode(next_state, parent=self, action=action)
        self.children.append(child)
        return child

def mcts(root_state, num_simulations=1000):
    root = MCTSNode(root_state)
    for _ in range(num_simulations):
        node = root
        while node.is_fully_expanded() and not node.state.is_terminal():
            node = node.best_child()
        if not node.state.is_terminal() and not node.is_fully_expanded():
            node = node.expand()
        sim_state = TicTacToe(node.state.board, node.state.player)
        while not sim_state.is_terminal():
            actions = sim_state.get_legal_actions(sim_state.current_player())
            action = random.choice(actions)
            sim_state = sim_state.get_successor(action)
        result = sim_state.utility(root_state.current_player())
        while node is not None:
            node.visits += 1
            node.wins += (result + 1) / 2
            node = node.parent
    return max(root.children, key=lambda n: n.visits).action

print('MCTS implemented')

In [ ]:
game_mcts = TicTacToe()
print('Testing MCTS...')
best_action = mcts(game_mcts, num_simulations=1000)
print(f'MCTS best move: {best_action}')

## 3.7 Summary

### Algorithm Comparison

| Algorithm | Type | Time | Best For |
|-----------|------|------|----------|
| Minimax | Deductive | O(b^d) | Small games |
| Alpha-Beta | Deductive | O(b^(d/2)) | Chess |
| MCTS | Inductive | Anytime | Go, large branching |

### Key Points
- Minimax assumes optimal opponents
- Alpha-Beta doubles effective search depth
- MCTS learns without domain knowledge
- Modern AI combines both approaches

---

**Next**: [Chapter 4 - Propositional Logic](ch04_propositional.ipynb)